# 06 - Hub Capacity & Performance Analysis

Investigating whether hub operating capacity correlates with delivery performance — testing the hypothesis that May–July 2024's dip was driven by operational stress on capacity-constrained hubs, particularly Dallas Main Hub, which operates significantly over stated capacity.


## Setup

In [1]:
import pandas as pd
import numpy as np
import sqlite3

df = pd.read_csv('/Users/DianaLara/PycharmProjects/PythonLearning_DataAnalysis/Orders.csv', encoding='UTF-16', sep='\t')
hubs_df = pd.read_csv('Hubs.csv', encoding='utf-16', sep='\t')

df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Actual Delivery Date'] = pd.to_datetime(df['Actual Delivery Date'], dayfirst=True, errors='coerce')
df['Order Month'] = df['Order Date'].dt.to_period('M')

df.head()


,Order ID,Actual Delivery Date,Delay Reason,Driver ID,Driver Name,Hub Name,Is Delayed,Is On Time,Order Date,Order Status,Vehicle Name,Vehicle Type,Customer Satisfaction Score,Delivery Time Hours,Hub Processing Time Hours,Order Month
0,1,2024-10-25,NaN,43,Karen Rodriguez,San Antonio Hub,False,True,2024-10-25,Delivered,FT-036,Truck,4,6.81,0.89,2024-10
1,2,2024-06-16,NaN,29,Matthew Williams,Houston Hub,False,True,2024-06-16,Delivered,FT-016,Van,4,5.74,3.60,2024-06
2,3,2024-07-05,NaN,25,Nancy Harris,Austin Hub,False,True,2024-07-05,Delivered,FT-040,Van,4,12.91,2.07,2024-07
3,4,2023-08-22,NaN,20,David Davis,Fort Worth Hub,False,True,2023-08-22,Delivered,FT-039,Van,5,9.40,2.37,2023-08
4,5,2024-06-06,Severe Weather,49,Joseph Williams,Dallas Main Hub,True,False,2024-06-02,Delivered,FT-018,Truck,3,103.48,1.80,2024-06


## 1. Hub Capacity Overview

Comparing stated hub capacity against actual order volume to calculate utilization rates.


In [2]:
# Total orders per hub across full dataset
hub_order_totals = df.groupby('Hub Name').size()

# Merge with capacity data
hub_analysis = hubs_df.copy()
hub_analysis['total_orders'] = hub_analysis['HubName'].map(hub_order_totals)
hub_analysis['avg_monthly_orders'] = hub_analysis['total_orders'] / 24
hub_analysis['utilization_pct'] = (hub_analysis['avg_monthly_orders'] / hub_analysis['Hub Capacity'] * 100).round(1)

print(hub_analysis[['HubName', 'Hub Capacity', 'avg_monthly_orders', 'utilization_pct']].to_string())


           HubName  Hub Capacity  avg_monthly_orders  utilization_pct
0  Dallas Main Hub           250          306.041667            122.4
1      Houston Hub           380          286.458333             75.4
2       Austin Hub           220          169.375000             77.0
3  San Antonio Hub           200          155.041667             77.5
4   Fort Worth Hub           180          135.625000             75.3
5      El Paso Hub           150          113.250000             75.5


**Finding:** Dallas Main Hub is the only hub operating significantly over stated capacity at 122% utilization (averaging 306 orders/month vs. 250 capacity). All other hubs sit in the 75-78% range, with Fort Worth being the most underutilized at 75%. If capacity is a constraint, Dallas Main should show worse performance metrics.


## 2. Performance by Hub Capacity Utilization

Testing the core hypothesis: does higher utilization correlate with worse delivery outcomes?


In [3]:
# Merge utilization back into main dataframe
df = df.merge(hub_analysis[['HubName', 'utilization_pct']], left_on='Hub Name', right_on='HubName', how='left')

# Group performance by utilization level
performance_by_utilization = df.groupby('Hub Name').agg({
    'Delivery Time Hours': 'mean',
    'Customer Satisfaction Score': 'mean',
    'Is On Time': lambda x: x.mean() * 100,
    'utilization_pct': 'first'
}).sort_values('utilization_pct', ascending=False)

performance_by_utilization.columns = ['avg_delivery_hours', 'avg_satisfaction', 'on_time_pct', 'utilization_pct']
print(performance_by_utilization.round(2))


                 avg_delivery_hours  avg_satisfaction  on_time_pct  \
Hub Name                                                             
Dallas Main Hub               35.78              4.17        78.91   
San Antonio Hub               35.79              4.16        78.69   
Austin Hub                    35.53              4.15        77.88   
El Paso Hub                   35.24              4.19        80.57   
Houston Hub                   35.79              4.17        79.08   
Fort Worth Hub                36.50              4.17        78.46   

                 utilization_pct  
Hub Name                          
Dallas Main Hub            122.4  
San Antonio Hub             77.5  
Austin Hub                  77.0  
El Paso Hub                 75.5  
Houston Hub                 75.4  
Fort Worth Hub              75.3  


**Finding:** examining the relationship between utilization and performance metrics. Look for whether Dallas Main (highest utilization) also shows the worst metrics.


In [4]:
# Calculate correlation between utilization and performance
correlation_utilization_delivery = df['utilization_pct'].corr(df['Delivery Time Hours'])
correlation_utilization_satisfaction = df['utilization_pct'].corr(df['Customer Satisfaction Score'])

print(f"Correlation: Utilization % vs. Delivery Time: {correlation_utilization_delivery:.3f}")
print(f"Correlation: Utilization % vs. Satisfaction: {correlation_utilization_satisfaction:.3f}")


Correlation: Utilization % vs. Delivery Time: -0.000
Correlation: Utilization % vs. Satisfaction: 0.002


## 3. Dallas Main Hub Deep-Dive

Since Dallas Main is the clear outlier in capacity utilization, testing whether this stress shows up in May-July 2024 specifically.


In [5]:
# Create flag for May-July 2024
df['is_may_july_2024'] = ((df['Order Month'] >= pd.Period('2024-05', freq='M')) & 
                          (df['Order Month'] <= pd.Period('2024-07', freq='M')))

dallas_main = df[df['Hub Name'] == 'Dallas Main Hub']

dallas_overall = dallas_main.groupby('is_may_july_2024').agg({
    'Delivery Time Hours': 'mean',
    'Customer Satisfaction Score': 'mean',
    'Is On Time': lambda x: x.mean() * 100
}).round(2)

dallas_overall.index = ['Rest of dataset', 'May-July 2024']
print("Dallas Main Hub performance: May-July 2024 vs. Rest")
print(dallas_overall)


Dallas Main Hub performance: May-July 2024 vs. Rest
                 Delivery Time Hours  Customer Satisfaction Score  Is On Time
Rest of dataset                35.76                         4.18       79.37
May-July 2024                  35.92                         4.12       76.06


**Finding:** compare Dallas Main's May-July 2024 metrics against its overall average to see if the dip was proportionally worse for the over-capacity hub.


In [6]:
# Compare Dallas Main's dip magnitude to other hubs
may_july_performance = df[df['is_may_july_2024'] == True].groupby('Hub Name').agg({
    'Delivery Time Hours': 'mean',
    'Customer Satisfaction Score': 'mean'
}).round(2)

overall_performance = df.groupby('Hub Name').agg({
    'Delivery Time Hours': 'mean',
    'Customer Satisfaction Score': 'mean'
}).round(2)

dip_magnitude = (may_july_performance - overall_performance).round(2)
dip_magnitude.columns = ['delivery_time_diff', 'satisfaction_diff']

print("\nMay-July 2024 dip magnitude by hub (compared to their own average):")
print(dip_magnitude.sort_values('delivery_time_diff', ascending=False))



May-July 2024 dip magnitude by hub (compared to their own average):
                 delivery_time_diff  satisfaction_diff
Hub Name                                              
El Paso Hub                    3.09              -0.05
Fort Worth Hub                 2.13              -0.07
San Antonio Hub                1.49              -0.06
Houston Hub                    0.49               0.01
Dallas Main Hub                0.14              -0.05
Austin Hub                    -0.08              -0.03


## 4. Reusable Function: Hub Capacity Performance Analysis

A function to analyze performance by utilization bucket, reusable across different thresholds or time periods.


In [7]:
def hub_capacity_kpi(df, group_by='Hub Name', period_filter=None):
    """Analyze hub performance metrics by hub, optionally filtered by time period.
    
    Args:
        df: dataframe with Hub Name, utilization_pct, and performance columns
        group_by: column to group by (default: 'Hub Name')
        period_filter: tuple of (bool_column, True/False) to filter by period
                      e.g. (df['is_may_july_2024'], True) for May-July only
    """
    if period_filter is not None:
        subset = df[period_filter[0] == period_filter[1]]
        period_name = 'May-July 2024' if period_filter[1] else 'Rest of dataset'
    else:
        subset = df
        period_name = 'Full dataset'
    
    kpi = subset.groupby(group_by).agg({
        'Order ID': 'count',
        'Delivery Time Hours': 'mean',
        'Customer Satisfaction Score': 'mean',
        'Is On Time': lambda x: x.mean() * 100,
        'utilization_pct': 'first'
    }).sort_values('utilization_pct', ascending=False)
    
    kpi.columns = ['total_orders', 'avg_delivery_hours', 'avg_satisfaction', 'on_time_pct', 'utilization_pct']
    kpi = kpi.round(2)
    
    print(f"\n{period_name}:")
    print(kpi)
    return kpi


In [8]:
# Test the function
full_period = hub_capacity_kpi(df)
may_july_only = hub_capacity_kpi(df, period_filter=(df['is_may_july_2024'], True))



Full dataset:
                 total_orders  avg_delivery_hours  avg_satisfaction  \
Hub Name                                                              
Dallas Main Hub          7345               35.78              4.17   
San Antonio Hub          3721               35.79              4.16   
Austin Hub               4065               35.53              4.15   
El Paso Hub              2718               35.24              4.19   
Houston Hub              6875               35.79              4.17   
Fort Worth Hub           3255               36.50              4.17   

                 on_time_pct  utilization_pct  
Hub Name                                       
Dallas Main Hub        78.91            122.4  
San Antonio Hub        78.69             77.5  
Austin Hub             77.88             77.0  
El Paso Hub            80.57             75.5  
Houston Hub            79.08             75.4  
Fort Worth Hub         78.46             75.3  

May-July 2024:
                

## Summary of Key Findings

**Hub Capacity Utilization**
- Dallas Main Hub operates at **122% capacity** (averaging 306 orders/month vs. 250 stated capacity), the only hub significantly over-capacity.
- All other hubs operate in the **75-78% range**, with Fort Worth most underutilized at 76%.

**Capacity vs. Performance Relationship**
- Test results will show whether higher utilization correlates with worse delivery time, lower satisfaction, or worse on-time rates.
- If correlation exists, this supports the hypothesis that May-July 2024's dip was operational stress from capacity constraints.
- If no correlation: capacity alone may not be the limiting factor; other operational factors may matter more.

**Dallas Main's May-July 2024 Performance**
- Compare Dallas Main's May-July 2024 metrics against its own average to see if the dip hit it harder than other hubs (expected if over-capacity amplifies stress).
- Dip magnitude across all hubs shows which ones felt May-July 2024 more acutely. If capacity is the driver, Dallas should rank worst.

**Next Steps:** if capacity utilization shows a clear relationship with performance, this is strong evidence that operational constraints drove the May-July 2024 dip; if not, investigate other operational factors (staffing, external events, route changes) that may have caused the slowdown.
